# Actividad 6 — Implementación de un evaluador de cadenas para un AFD y un AFN

**Materia:** Lenguajes de Computación · Otoño 2026

| Nombre completo | No. de cuenta |
|---|---|
| Angel Rugerio Jiménez | 201720 |
| Axel García Arellano | 201251 |

## Objetivo

Implementar un evaluador de AFD y uno de AFN a través de un servicio web, donde **cada evaluador
cuenta con su propio endpoint**.

## Qué hace el servicio

El archivo [`main.py`](./main.py) levanta una aplicación de [FastAPI](https://fastapi.tiangolo.com/)
con dos endpoints:

| Endpoint | Autómata | Transición |
|---|---|---|
| `POST /afd/evaluar` | Autómata finito **determinista** | $\delta: Q \times \Sigma \to Q$ |
| `POST /afn/evaluar` | Autómata finito **no determinista** | $\Delta: Q \times (\Sigma \cup \{\varepsilon\}) \to \mathcal{P}(Q)$ |

Los dos reciben `tabla_transicion`, `estado_inicial`, `estados_finales` y las `cadenas` a evaluar,
y devuelven los estados totales $Q$, el alfabeto $\Sigma$, el estado inicial $s$, los estados
finales $F$ y el resultado de cada cadena con su **notación de transición**.

La única diferencia está en la forma de cada celda de la tabla: en el AFD es **un** estado
(`"q1"`) y en el AFN es un **conjunto** de estados (`["q0", "q1"]`).

Los dos autómatas que se evalúan ya los habíamos trabajado a mano: el AFD es el del **Reporte 1**
y el AFN es el de la **Actividad 5**, con las mismas cadenas, así que sirven para comprobar que el
evaluador da lo mismo que salió en papel.

---
## 1. Levantar el servicio

Se arranca `uvicorn` en un subproceso para poder hacer peticiones HTTP **reales** a los endpoints.
No se importan las funciones de `main.py` directamente: todos los resultados de esta libreta salen
del servicio web.

In [1]:
import json
import subprocess
import sys
import time

import httpx

BASE = "http://127.0.0.1:8000"

servidor = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Esperar a que el servidor conteste antes de seguir
for _ in range(60):
    try:
        httpx.get(BASE + "/", timeout=1)
        break
    except Exception:
        time.sleep(0.5)

print("Servicio arriba:", httpx.get(BASE + "/").json()["actividad"])

Servicio arriba: Actividad 6 - Evaluador de cadenas para AFD y AFN


### Funciones auxiliares

`evaluar()` manda el autómata al endpoint y las funciones `mostrar_*` le dan formato a la respuesta
para poder leerla aquí.

In [2]:
AUTOMATAS = json.load(open("automatas.json", encoding="utf-8"))


def evaluar(endpoint: str, automata: dict) -> dict:
    """Manda el automata al endpoint indicado y regresa el JSON de respuesta."""
    cuerpo = {
        "estado_inicial": automata["estado_inicial"],
        "estados_finales": automata["estados_finales"],
        "tabla_transicion": automata["tabla_transicion"],
        "cadenas": automata["cadenas"],
    }
    respuesta = httpx.post(BASE + endpoint, json=cuerpo, timeout=30)
    respuesta.raise_for_status()
    return respuesta.json()


def celda(valor) -> str:
    """Una celda de la tabla: un estado en el AFD, un conjunto en el AFN."""
    if valor is None:
        return "Φ"
    if isinstance(valor, str):
        return valor
    return "{" + ",".join(valor) + "}"


def mostrar_tabla(automata: dict, delta: str = "δ") -> None:
    """Tabla de transiciones como en clase: filas = simbolos, columnas = estados."""
    tabla = automata["tabla_transicion"]
    estados = sorted(tabla)
    alfabeto = sorted({s for q in tabla for s in tabla[q]})

    def titulo(q):
        return ("->" if q == automata["estado_inicial"] else "") + \
               ("*" if q in automata["estados_finales"] else "") + q

    anchos = [len(titulo(q)) for q in estados]
    anchos += [len(celda(tabla[q].get(s))) for q in estados for s in alfabeto]
    ancho = max(anchos) + 3
    print(f"{delta:<4}" + "".join(f"{titulo(q):<{ancho}}" for q in estados))
    print("-" * (4 + ancho * len(estados)))
    for s in alfabeto:
        print(f"{s:<4}" + "".join(f"{celda(tabla[q].get(s)):<{ancho}}" for q in estados))


def mostrar_definicion(r: dict) -> None:
    """La quintupla A = (Q, Σ, δ, s, F) que devolvio el endpoint."""
    print(f"Tipo : {r['tipo']}   {r['definicion_formal']}")
    print(f"Q    : {{{', '.join(r['estados_totales'])}}}   -> {r['numero_de_estados']} estados")
    print(f"Σ    : {{{', '.join(r['alfabeto'])}}}")
    print(f"s    : {r['estado_inicial']}")
    print(f"F    : {{{', '.join(r['estados_finales'])}}}")


def mostrar_resultados(r: dict) -> None:
    """Una linea por cadena evaluada: veredicto y donde termino el automata."""
    ancho = max(8, max(len(x["cadena"]) for x in r["resultados"]) + 2)
    print(f"{'CADENA':<{ancho}}{'RESULTADO':<12}TERMINA EN")
    print("-" * (ancho + 32))
    for x in r["resultados"]:
        if "estado_final_alcanzado" in x:          # AFD
            fin = x["estado_final_alcanzado"] or "(se detuvo antes)"
        else:                                      # AFN
            fin = celda(x["configuracion_final"]) if x["configuracion_final"] else "Φ"
        print(f"{x['cadena']:<{ancho}}{'VALIDA' if x['aceptada'] else 'invalida':<12}{fin}")
    print()
    print(f"Validas   ({len(r['resumen']['aceptadas'])}): {', '.join(r['resumen']['aceptadas'])}")
    print(f"Invalidas ({len(r['resumen']['rechazadas'])}): {', '.join(r['resumen']['rechazadas'])}")


def mostrar_traza(r: dict, cadena: str) -> None:
    """Notacion de transicion completa de una sola cadena."""
    x = next(res for res in r["resultados"] if res["cadena"] == (cadena or "ε"))
    notacion = x["notacion_transicion"]

    print(f"Cadena '{x['cadena']}'  ->  {'ACEPTADA' if x['aceptada'] else 'RECHAZADA'}\n")
    for paso in notacion["pasos"]:
        print("    " + paso)

    recorrido = notacion.get("recorrido") or notacion.get("camino_de_aceptacion")
    if recorrido:
        print("\n    Recorrido:  " + recorrido)
    print("\n    " + x["motivo"])

---
## 2. El servicio

`GET /` describe el servicio. FastAPI genera además la documentación interactiva en `/docs`.

In [3]:
print(json.dumps(httpx.get(BASE + "/").json(), indent=2, ensure_ascii=False))

{
  "actividad": "Actividad 6 - Evaluador de cadenas para AFD y AFN",
  "equipo": [
    "Angel Rugerio Jimenez - 201720",
    "Axel Garcia Arellano - 201251"
  ],
  "endpoints": {
    "POST /afd/evaluar": "Evalua cadenas sobre un AFD",
    "POST /afn/evaluar": "Evalua cadenas sobre un AFN (con o sin transiciones ε)"
  },
  "documentacion_interactiva": "/docs"
}


---
## 3. Ejercicio 1 — Evaluador de **AFD**

Se usa el AFD del **Reporte 1**, sobre $\Sigma = \{a, b, c, d, e\}$ y con 10 estados. Es un AFD
**completo**: hay exactamente una transición por cada par (estado, símbolo). El estado $q_9$ es el
**estado trampa**, absorbente y no final, así que cualquier cadena con el símbolo `e` termina
rechazada.

Las 15 cadenas son las mismas que se verificaron a mano en el reporte.

In [4]:
afd = AUTOMATAS["ejercicio_1_afd"]
print(afd["nombre"], "\n")
mostrar_tabla(afd)

AFD del Reporte 1 (Σ = {a, b, c, d, e}, 10 estados) 

δ   ->q0   q1     *q2    q3     *q4    q5     *q6    *q7    *q8    q9     
--------------------------------------------------------------------------
a   q1     q1     q4     q1     q4     q6     q6     q4     q6     q9     
b   q2     q2     q2     q6     q2     q2     q7     q7     q2     q9     
c   q3     q4     q6     q3     q7     q3     q8     q8     q8     q9     
d   q5     q5     q5     q7     q8     q5     q5     q7     q8     q9     
e   q9     q9     q9     q9     q9     q9     q9     q9     q9     q9     


### 3.1 Petición a `POST /afd/evaluar`

Este es el cuerpo JSON que se envía (recortado a las primeras filas para que se lea):

In [5]:
cuerpo = {
    "estado_inicial": afd["estado_inicial"],
    "estados_finales": afd["estados_finales"],
    "tabla_transicion": afd["tabla_transicion"],
    "cadenas": afd["cadenas"],
}
print(json.dumps(cuerpo, indent=2, ensure_ascii=False)[:600] + "\n   ... (resto de la tabla) ...")

{
  "estado_inicial": "q0",
  "estados_finales": [
    "q2",
    "q4",
    "q6",
    "q7",
    "q8"
  ],
  "tabla_transicion": {
    "q0": {
      "a": "q1",
      "b": "q2",
      "c": "q3",
      "d": "q5",
      "e": "q9"
    },
    "q1": {
      "a": "q1",
      "b": "q2",
      "c": "q4",
      "d": "q5",
      "e": "q9"
    },
    "q2": {
      "a": "q4",
      "b": "q2",
      "c": "q6",
      "d": "q5",
      "e": "q9"
    },
    "q3": {
      "a": "q1",
      "b": "q6",
      "c": "q3",
      "d": "q7",
      "e": "q9"
    },
    "q4": {
      "a": "q4",
      "b": "q2",
      "c": "q
   ... (resto de la tabla) ...


In [6]:
r_afd = evaluar("/afd/evaluar", afd)
mostrar_definicion(r_afd)

Tipo : AFD   A = (Q, Σ, δ, s, F)  con  δ: Q × Σ → Q
Q    : {q0, q1, q2, q3, q4, q5, q6, q7, q8, q9}   -> 10 estados
Σ    : {a, b, c, d, e}
s    : q0
F    : {q2, q4, q6, q7, q8}


Coincide con lo que habíamos identificado en el Reporte 1: $Q$ con 10 estados,
$\Sigma = \{a,b,c,d,e\}$, $s = q_0$ y $F = \{q_2, q_4, q_6, q_7, q_8\}$.

### 3.2 Resultado de cada cadena

In [7]:
mostrar_resultados(r_afd)

CADENA  RESULTADO   TERMINA EN
----------------------------------------
aaaab   VALIDA      q2
cac     VALIDA      q4
b       VALIDA      q2
abccc   VALIDA      q8
ab      VALIDA      q2
aac     VALIDA      q4
cccc    invalida    q3
e       invalida    q9
aaccc   VALIDA      q8
bbc     VALIDA      q6
d       invalida    q5
cdddd   VALIDA      q7
ad      invalida    q5
bbd     invalida    q5
bbbbb   VALIDA      q2

Validas   (10): aaaab, cac, b, abccc, ab, aac, aaccc, bbc, cdddd, bbbbb
Invalidas (5): cccc, e, d, ad, bbd


Las 15 cadenas dan exactamente el mismo veredicto que salió a mano en el reporte.

### 3.3 Notación de transición

Tres casos representativos: una cadena aceptada, una rechazada por terminar en un estado no final,
y una rechazada por caer en el estado trampa $q_9$.

In [8]:
mostrar_traza(r_afd, "abccc")

Cadena 'abccc'  ->  ACEPTADA

    δ(q0, a) = q1
    δ(q1, b) = q2
    δ(q2, c) = q6
    δ(q6, c) = q8
    δ(q8, c) = q8

    Recorrido:  q0 --a--> q1 --b--> q2 --c--> q6 --c--> q8 --c--> q8

    La cadena se consumio por completo y termino en 'q8', que SI es un estado final.


In [9]:
mostrar_traza(r_afd, "cccc")

Cadena 'cccc'  ->  RECHAZADA

    δ(q0, c) = q3
    δ(q3, c) = q3
    δ(q3, c) = q3
    δ(q3, c) = q3

    Recorrido:  q0 --c--> q3 --c--> q3 --c--> q3 --c--> q3

    La cadena se consumio por completo y termino en 'q3', que NO es un estado final.


In [10]:
mostrar_traza(r_afd, "e")

Cadena 'e'  ->  RECHAZADA

    δ(q0, e) = q9

    Recorrido:  q0 --e--> q9

    La cadena se consumio por completo y termino en 'q9', que NO es un estado final.


---
## 4. Ejercicio 2 — Evaluador de **AFN**

Se usa el AFN de la **Actividad 5**, sobre $\Sigma = \{a, b, c\}$. El no determinismo está en
$q_0$: al leer `a` el autómata **se queda en $q_0$ y a la vez pasa a $q_1$**, es decir
$\Delta(q_0, a) = \{q_0, q_1\}$.

Nótese que la tabla ya no tiene un estado por celda sino un **conjunto**, y que hay celdas vacías
($\Phi$), algo imposible en un AFD completo.

In [11]:
afn = AUTOMATAS["ejercicio_2_afn"]
print(afn["nombre"], "\n")
mostrar_tabla(afn, delta="Δ")

AFN de la Actividad 5 (Σ = {a, b, c}, 6 estados) 

Δ   ->q0      q1        q2        q3        *q4       *q5       
----------------------------------------------------------------
a   {q0,q1}   Φ         Φ         Φ         Φ         Φ         
b   {q0}      {q2}      Φ         {q5}      Φ         Φ         
c   {q0}      {q3}      {q4}      Φ         Φ         Φ         


In [12]:
r_afn = evaluar("/afn/evaluar", afn)
mostrar_definicion(r_afn)

Tipo : AFN   A = (Q, Σ, Δ, s, F)  con  Δ: Q × (Σ ∪ {ε}) → P(Q)
Q    : {q0, q1, q2, q3, q4, q5}   -> 6 estados
Σ    : {a, b, c}
s    : q0
F    : {q4, q5}


### 4.1 Resultado de cada cadena

Son las 9 cadenas de la Actividad 5. La columna "termina en" ya no es un estado sino la
**configuración final**: el conjunto de estados en los que el autómata podría estar al acabar la
cadena. Se acepta si ese conjunto toca a $F$.

In [13]:
mostrar_resultados(r_afn)

CADENA  RESULTADO   TERMINA EN
----------------------------------------
abc     VALIDA      {q0,q4}
acb     VALIDA      {q0,q5}
ab      invalida    {q0,q2}
ac      invalida    {q0,q3}
cbaacb  VALIDA      {q0,q5}
bbabc   VALIDA      {q0,q4}
abca    invalida    {q0,q1}
abcb    invalida    {q0}
accb    invalida    {q0}

Validas   (4): abc, acb, cbaacb, bbabc
Invalidas (5): ab, ac, abca, abcb, accb


### 4.2 Notación de transición

El evaluador rastrea la **configuración** (el conjunto de estados activos), así que cada paso tiene
la forma $\Delta(\{q_i, \dots\}, a) = \{q_j, \dots\}$. Cuando la cadena se acepta se busca además
**un recorrido concreto** que la acepte, que es el que se escribió a mano en la Actividad 5.

In [14]:
mostrar_traza(r_afn, "abc")

Cadena 'abc'  ->  ACEPTADA

    Δ({q0}, a) = {q0, q1}
    Δ({q0, q1}, b) = {q0, q2}
    Δ({q0, q2}, c) = {q0, q4}

    Recorrido:  q0 --a--> q1 --b--> q2 --c--> q4

    La configuracion final es {q0, q4} y contiene {q4} de F, asi que existe al menos un recorrido que acepta la cadena.


En el primer paso se ve el no determinismo: con `a` desde $\{q_0\}$ se llega a $\{q_0, q_1\}$, o
sea la rama que sigue leyendo prefijo y la que apuesta a que ahí empieza el sufijo. El recorrido
que acepta es $q_0 \xrightarrow{a} q_1 \xrightarrow{b} q_2 \xrightarrow{c} q_4$, igual que en la
Actividad 5.

Una cadena más larga, donde el autómata consume prefijo antes de acertar:

In [15]:
mostrar_traza(r_afn, "cbaacb")

Cadena 'cbaacb'  ->  ACEPTADA

    Δ({q0}, c) = {q0}
    Δ({q0}, b) = {q0}
    Δ({q0}, a) = {q0, q1}
    Δ({q0, q1}, a) = {q0, q1}
    Δ({q0, q1}, c) = {q0, q3}
    Δ({q0, q3}, b) = {q0, q5}

    Recorrido:  q0 --c--> q0 --b--> q0 --a--> q0 --a--> q1 --c--> q3 --b--> q5

    La configuracion final es {q0, q5} y contiene {q5} de F, asi que existe al menos un recorrido que acepta la cadena.


Y un caso rechazado: en `abcb` la rama buena llega a $q_4$ pero ahí se le acaba la salida (la rama
muere, $\Phi$), y la única rama que sobrevive es la que se quedó dando vueltas en $q_0$.

In [16]:
mostrar_traza(r_afn, "abcb")

Cadena 'abcb'  ->  RECHAZADA

    Δ({q0}, a) = {q0, q1}
    Δ({q0, q1}, b) = {q0, q2}
    Δ({q0, q2}, c) = {q0, q4}
    Δ({q0, q4}, b) = {q0}

    La configuracion final es {q0} y ninguno de esos estados es final.


---
## 5. Casos borde y validación de la entrada

### 5.1 AFD con tabla incompleta (estado trampa implícito)

Si la tabla no define $\delta(q, a)$ para algún par, el evaluador no truena: detiene el recorrido e
informa en qué símbolo se quedó, que es lo mismo que caer en un estado trampa.

In [17]:
parcial = {
    "estado_inicial": "q0",
    "estados_finales": ["q2"],
    "tabla_transicion": {"q0": {"a": "q1"}, "q1": {"b": "q2"}, "q2": {}},
    "cadenas": ["ab", "aa", "abb"],
}
r_parcial = evaluar("/afd/evaluar", parcial)
mostrar_resultados(r_parcial)
print()
mostrar_traza(r_parcial, "aa")

CADENA  RESULTADO   TERMINA EN
----------------------------------------
ab      VALIDA      q2
aa      invalida    (se detuvo antes)
abb     invalida    (se detuvo antes)

Validas   (1): ab
Invalidas (2): aa, abb

Cadena 'aa'  ->  RECHAZADA

    δ(q0, a) = q1

    Recorrido:  q0 --a--> q1

    No existe la transicion δ(q1, a): el AFD se detiene en el simbolo 2 de la cadena (equivale a caer en un estado trampa).


### 5.2 Errores en la definición del autómata

El servicio responde `400` si el estado inicial o alguno de los finales no aparece en la tabla de
transición. Esto funciona porque $Q$ se deduce **solo** de la tabla (renglones y destinos), nunca
de `estado_inicial` ni de `estados_finales`: así un estado mal escrito se detecta en vez de
agregarse en silencio al autómata.

In [18]:
malos = [
    ("estado inicial inexistente",
     {"estado_inicial": "qX", "estados_finales": ["q2"],
      "tabla_transicion": {"q0": {"a": "q1"}, "q1": {"b": "q2"}, "q2": {}}, "cadenas": ["ab"]}),
    ("estado final inexistente",
     {"estado_inicial": "q0", "estados_finales": ["q9"],
      "tabla_transicion": {"q0": {"a": "q1"}, "q1": {"b": "q2"}, "q2": {}}, "cadenas": ["ab"]}),
]
for descripcion, cuerpo_malo in malos:
    resp = httpx.post(BASE + "/afd/evaluar", json=cuerpo_malo, timeout=10)
    print(f"{descripcion:<28} -> HTTP {resp.status_code}: {resp.json()['detail']}")

estado inicial inexistente   -> HTTP 400: El estado inicial 'qX' no aparece en el automata.
estado final inexistente     -> HTTP 400: Estados finales que no aparecen en el automata: ['q9']


### 5.3 El endpoint de AFD rechaza tablas no deterministas

El tipo declarado del AFD es `Dict[str, Dict[str, str]]`, así que si una celda trae un conjunto de
estados Pydantic contesta `422` y obliga a usar el endpoint de AFN. La forma de la tabla es
suficiente para distinguir los dos autómatas.

In [19]:
no_determinista = {
    "estado_inicial": "q0", "estados_finales": ["q1"],
    "tabla_transicion": {"q0": {"a": ["q0", "q1"]}, "q1": {}}, "cadenas": ["a"],
}
resp = httpx.post(BASE + "/afd/evaluar", json=no_determinista, timeout=10)
print("POST /afd/evaluar ->", resp.status_code, resp.json()["detail"][0]["msg"])

resp = httpx.post(BASE + "/afn/evaluar", json=no_determinista, timeout=10)
print("POST /afn/evaluar ->", resp.status_code, "|",
      [(x["cadena"], x["aceptada"]) for x in resp.json()["resultados"]])

POST /afd/evaluar -> 422 Input should be a valid string
POST /afn/evaluar -> 200 | [('a', True)]


---
## 6. Conclusiones

* Los dos evaluadores comparten la lectura de la tabla de transición (de ahí salen $Q$ y $\Sigma$),
  pero difieren en la simulación: el AFD mantiene **un estado actual** y el AFN mantiene una
  **configuración**, o sea un conjunto de estados activos.
* La forma de la tabla basta para distinguir los dos autómatas: una celda con un estado (AFD)
  contra una celda con un conjunto de estados (AFN). Por eso el endpoint de AFD puede rechazar solo
  una tabla no determinista (§5.3).
* Un AFD acepta una cadena si su **único** recorrido termina en un estado final; un AFN la acepta si
  **existe al menos un** recorrido que lo haga, aunque las otras ramas mueran o terminen en estados
  no finales. Se ve en `abcb` (§4.2): la rama que llegó a $q_4$ muere y la que sobrevive no es
  final, así que la cadena se rechaza.
* Los dos ejercicios dan exactamente el mismo veredicto que habíamos sacado a mano en el Reporte 1
  y en la Actividad 5, que era el punto de comprobación.

In [20]:
servidor.terminate()
servidor.wait(timeout=10)
print("Servicio detenido.")

Servicio detenido.
